# Lab: Understanding AI Vector Comparison with Milvus

In this lab, we will explore how AI agents understand and compare text. Instead of reading words like humans do, AI converts text into dense numerical vectors called **Embeddings**. 

By calculating the mathematical distance between these vectors, an AI can determine how semantically similar two pieces of text are. We will use **Milvus**, a specialized vector database, to perform these comparisons efficiently.

## Objectives:
1. Understand how an AI compares short sentences using embeddings and Milvus.
2. Observe the actual similarity metrics computed under the hood.
3. Scale up the comparison to search through an entire corpus of documents (reusing the collection built in the RAG demo).

Don't forget to RTFM

> ⚠️ **IMPORTANT: "Database is locked" Error**
> 
> Milvus Lite stores its data in a local file (e.g., `milvus_demo.db`). If you still have the previous notebooks (`tp1-simple-demo.ipynb` or `tp2-RAG.ipynb`) open and running in the background, they might hold a lock on this file. 
> 
> If you encounter an error saying **"Open /home/jovyan/milvus_demo.db failed"**:
> 1. Go to the **"Running"** tab (or the Kernel/Terminal manager) in your Jupyter interface.
> 2. **Shut down** the kernels for TP1 and TP2.

---

## Part 1: Comparing Sentences via Vector Distance

First, let's look at how an AI determines if two sentences mean the same thing. We will use an embedding model to convert three sentences into vectors. Then, we will insert them into a temporary Milvus collection and use Milvus to search for the most similar sentence.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from pymilvus import MilvusClient

# Load environment variables
load_dotenv()

# Initialize the embedding model using the credentials in your .env file
embeddings = OpenAIEmbeddings(
    model=os.getenv("AI_EMBEDDING_MODEL"),
    base_url=os.getenv("AI_ENDPOINT"),
    api_key=os.getenv("AI_API_KEY")
)

# Connect to the local Milvus database
milvus_client = MilvusClient(uri="./milvus_demo.db")
collection_name = "sentence_comparison_demo"

# Clean up previous runs if necessary
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

# Here are three sentences. Two are related, one is completely different.
sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "A fast dark-colored fox leaps above a sleepy hound.",
    "Quantum computing leverages the principles of quantum mechanics."
]

print("Generating embeddings for the sentences...")
embedded_data = []

# TODO 1: Generate embeddings for each sentence.
# Hint: Use `embeddings.embed_query(text)` to convert a string to a vector.
for i, sentence in enumerate(sentences):
    # YOUR CODE HERE
    # vector = ...
    
    # We store the ID, the vector, and the original text
    # embedded_data.append({"id": i, "vector": vector, "text": sentence})
    pass

print("Successfully generated vectors!")

In [ ]:
# Create a temporary collection to hold our 3 sentences
# We need to specify the dimension of our vectors.
vector_dimension = len(embedded_data[0]['vector']) if embedded_data else 1536

milvus_client.create_collection(
    collection_name=collection_name,
    dimension=vector_dimension,
    metric_type="IP" # Inner Product measures similarity
)

# TODO 2: Insert the `embedded_data` into the Milvus collection.
# Hint: Use `milvus_client.insert(collection_name=..., data=...)`

# YOUR CODE HERE


# Now, let's search for the sentences most similar to our query.
# We will use the vector of the first sentence ("The quick brown fox...") as our search query.
query_vector = embedded_data[0]['vector'] if embedded_data else []

# TODO 3: Perform a search in Milvus using `query_vector`.
# Retrieve all 3 results (limit=3) to see the comparison scores for EVERY sentence.
# Make sure to include "text" in the output_fields.
# Hint: Use `milvus_client.search(...)`

# YOUR CODE HERE
# search_results = ...


print(f"\n🔍 Query Vector matches for: '{sentences[0]}'")
print("-" * 70)

# Let's peek under the hood and look at the actual similarity scores!
# Milvus returns a 'distance' metric. Since we used "IP" (Inner Product), a higher score means higher similarity.
try:
    if search_results:
        # search_results is a list of lists
        for hits in search_results:
            for hit in hits:
                # Extracting the distance metric and the original text
                score = hit['distance']
                text = hit['entity']['text']
                
                # Formatting the output to clearly show what happens under the hood
                print(f"Similarity Score: {score:.4f} --> {text}")
    else:
        print("No results to display. Complete the TODO first!")
except NameError:
    print("Variable 'search_results' is not defined. Complete the TODO first!")

---

## Part 2: Cross-Directory Document Comparison

Vector databases are great for **finding similarities at scale**, such as detecting plagiarism, finding duplicate files, or linking related articles. In this part, we will compare two entire directories of text files to find the top 2 most similar file pairs across the directories.

First, let's generate some dummy text files in two separate directories (`dir_A` and `dir_B`).

In [ ]:
import os
import shutil

# 1. Setup dummy directories and files
dir_a_path = "./dir_A"
dir_b_path = "./dir_B"

# Clean up if they already exist
for d in [dir_a_path, dir_b_path]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d)

# Create files for Directory A (The "Database")
docs_A = {
    "docA_1.txt": "Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability.",
    "docA_2.txt": "The James Webb Space Telescope is a space telescope designed primarily to conduct infrared astronomy.",
    "docA_3.txt": "Photosynthesis is a process used by plants and other organisms to convert light energy into chemical energy."
}

# Create files for Directory B (The "New Files" we want to check)
docs_B = {
    "docB_1.txt": "Plants absorb sunlight and turn it into energy they can use to grow, using chlorophyll.",
    "docB_2.txt": "Electric vehicles (EVs) are cars that run on electricity stored in batteries instead of gasoline.",
    "docB_3.txt": "NASA's newest telescope observes the universe in the infrared spectrum to see distant galaxies."
}

for filename, content in docs_A.items():
    with open(os.path.join(dir_a_path, filename), "w") as f: f.write(content)

for filename, content in docs_B.items():
    with open(os.path.join(dir_b_path, filename), "w") as f: f.write(content)

print("Directories 'dir_A' and 'dir_B' created with text files.")

Now, your mission is to embed the files from `dir_A` and store them in a new Milvus collection. Then, read `dir_B`, embed its files, and search Milvus to find which files in `dir_A` they match with.

In [ ]:
# Create a new collection for Directory A
collection_A_name = "dir_a_collection"
if milvus_client.has_collection(collection_A_name):
    milvus_client.drop_collection(collection_A_name)

# Notice we use the same embedding model from Part 1
vector_dimension = 1536 

milvus_client.create_collection(
    collection_name=collection_A_name,
    dimension=vector_dimension,
    metric_type="IP"
)

# 1. Read and Embed Directory A
embedded_A = []
for i, filename in enumerate(os.listdir(dir_a_path)):
    with open(os.path.join(dir_a_path, filename), "r") as f:
        text = f.read()
        # Embed the text
        vector = embeddings.embed_query(text)
        # Add filename as metadata so we know which file it is!
        embedded_A.append({"id": i, "vector": vector, "text": text, "filename": filename})

# TODO 4: Insert the `embedded_A` data into `collection_A_name` using `milvus_client`.
# YOUR CODE HERE


# 2. Compare Directory B against Directory A
all_comparisons = []

for filename in os.listdir(dir_b_path):
    with open(os.path.join(dir_b_path, filename), "r") as f:
        text_b = f.read()
        
        # TODO 5: Embed the text from Directory B.
        # YOUR CODE HERE
        # vector_b = ...

        
        # TODO 6: Search the Milvus collection to find the SINGLE most similar file in Dir A (limit=1).
        # We need the "filename" and "text" fields in the output.
        # YOUR CODE HERE
        # search_res = ...

        
        if search_res and search_res[0]:
            best_match = search_res[0][0]
            distance = best_match['distance']
            matched_filename = best_match['entity']['filename']
            
            # Store the comparison
            all_comparisons.append({
                "file_B": filename,
                "file_A": matched_filename,
                "score": distance,
                "text_B": text_b,
                "text_A": best_match['entity']['text']
            })

# 3. Sort by similarity score (highest first for Inner Product) and get the top 2
all_comparisons.sort(key=lambda x: x["score"], reverse=True)
top_2_pairs = all_comparisons[:2]

print("\\n🏆 Top 2 most similar file pairs between Dir B and Dir A:")
print("=" * 80)
for i, pair in enumerate(top_2_pairs):
    print(f"Rank {i+1} - Score: {pair['score']:.4f}")
    print(f"📁 Dir B: {pair['file_B']} -> '{pair['text_B']}'")
    print(f"📂 Dir A: {pair['file_A']} -> '{pair['text_A']}'")
    print("-" * 80)